# CNN v2 — Option A (Calibration) + Option B (Zero-shot)

Standard 2D CNN on raw sEMG (1, 8, 50). Single model, 5 scenario evaluations.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from config import RANDOM_SEED, N_CLASSES, MODELS_DIR, get_device
from src.experiment_runner import (
    get_splits, load_and_norm, split_cal_test,
    run_zero_shot, run_calibration, print_comparison,
    TEST_SUBJECTS, TRAIN_SUBJECTS,
)
from src.evaluation import measure_latency, print_latency

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = get_device()
splits = get_splits()
print(f'Device: {DEVICE}')

Device: mps


## Model

In [2]:
class StandardCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, n_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))
    def extract_feat(self, x):
        with torch.no_grad():
            f = self.features(x)
            return nn.Flatten()(f)

print(f'Params: {sum(p.numel() for p in StandardCNN().parameters()):,}')

Params: 101,831


## Training

In [3]:
train_combined = pd.concat([splits['train_df'], splits['s5_train']])
X_train, y_train, norm_stats = load_and_norm(train_combined, verbose=True)
print(f'Train: {X_train.shape}')

X_t = torch.from_numpy(X_train).float().unsqueeze(1)
y_t = torch.from_numpy(y_train).long()
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=256, shuffle=True)

model = StandardCNN().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)
crit = nn.CrossEntropyLoss()

for ep in range(30):
    model.train()
    loss_sum, correct, total = 0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = model(xb); loss = crit(out, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        loss_sum += loss.item()*xb.size(0)
        correct += (out.argmax(1)==yb).sum().item(); total += xb.size(0)
    sched.step()
    if (ep+1)%5==0 or ep==0:
        print(f'Epoch {ep+1:2d}/30 — loss:{loss_sum/total:.4f} acc:{correct/total:.4f}')

torch.save(model.state_dict(), MODELS_DIR / 'cnn_v2.pt')
print('Saved.')

Loading windows: 100%|██████████| 5790/5790 [00:02<00:00, 2752.07it/s]


Train: (651972, 8, 50)
Epoch  1/30 — loss:1.2634 acc:0.5242
Epoch  5/30 — loss:0.9145 acc:0.6683
Epoch 10/30 — loss:0.8081 acc:0.7081
Epoch 15/30 — loss:0.7329 acc:0.7368
Epoch 20/30 — loss:0.7111 acc:0.7442
Epoch 25/30 — loss:0.6726 acc:0.7591
Epoch 30/30 — loss:0.6630 acc:0.7618
Saved.


## Predict + feature extraction helpers

In [4]:
@torch.no_grad()
def cnn_predict(X):
    model.eval()
    X_t = torch.from_numpy(X).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(X_t), batch_size=512, shuffle=False)
    preds = []
    for (xb,) in loader:
        preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(preds)

@torch.no_grad()
def cnn_features(X):
    model.eval()
    X_t = torch.from_numpy(X).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(X_t), batch_size=512, shuffle=False)
    feats = []
    for (xb,) in loader:
        feats.append(model.extract_feat(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(feats)

def cnn_finetune(X_cal, y_cal):
    F_cal = cnn_features(X_cal)
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=1.0)
    clf.fit(F_cal, y_cal)
    def predict_ft(X):
        return clf.predict(cnn_features(X))
    return predict_ft

## Option B — Zero-shot

In [5]:
print('Option B — Zero-shot:')
zero_results = run_zero_shot(cnn_predict, splits, norm_stats, name='CNN')

Option B — Zero-shot:
  S1 zero-shot: 0.5836
  S2 zero-shot: 0.5492
  S3 zero-shot: 0.5523
  S4 zero-shot: 0.6657
  S5 zero-shot: 0.8129


## Option A — Calibration

Extract features from pre-trained CNN, train LogisticRegression on calibration data.

In [6]:
print('Option A — Calibration:')
cal_results = run_calibration(cnn_predict, cnn_finetune, splits, norm_stats, name='CNN')

Option A — Calibration:
  S1 calibrated: 0.7268
  S2 calibrated: 0.7526
  S3 calibrated: 0.7887
  S4 calibrated: 0.7941
  S5 calibrated: 0.8548


## Latency

In [7]:
model.eval()
s = torch.randn(1,1,8,50).to(DEVICE)
for _ in range(10): _ = model(s)
if DEVICE.type=='mps': torch.mps.synchronize()
def cnn_single(x):
    xt = torch.from_numpy(x).float().unsqueeze(1).to(DEVICE)
    with torch.no_grad(): out = model(xt)
    if DEVICE.type=='mps': torch.mps.synchronize()
    return out.argmax(1).cpu().numpy()
latency = measure_latency(cnn_single, X_train[:1], n_runs=500)
print_latency(latency, 'CNN')


Latency — CNN
  Mean:   5.25 ms
  Median: 4.20 ms
  P95:    11.50 ms
  <300ms: ✓


## Results

In [8]:
print_comparison(zero_results, cal_results, name='CNN (2D)')


  CNN (2D) — RESULTS
Scenario        Zero-shot   Calibrated        Δ
-------------------------------------------------------
S1                58.36%       72.68%   +14.32%
S2                54.92%       75.26%   +20.34%
S3                55.23%       78.87%   +23.64%
S4                66.57%       79.41%   +12.83%
S5                81.29%       85.48% ✓   +4.19%
